# Taller de herramientas: scikit-learn y NVIDIA NIM

**SI3003 · Sesiones 6–9 · 110–150 minutos incluyendo las preguntas conceptuales.**

Complete los `TODO`, ejecute sus experimentos y justifique los resultados. `...` marca una expresión pendiente. No implemente algoritmos desde cero: utilice los métodos de las bibliotecas. Las particiones y la conexión a NVIDIA están proporcionadas. Puede usar Google Colab con CPU.

In [ ]:
%pip -q install scikit-learn openai

## 1. Clasificación con scikit-learn

Iris contiene 150 flores de tres especies, representadas por cuatro medidas. Reservamos datos para entrenar, elegir entre dos profundidades con validación y evaluar finalmente en prueba. Ejecute esta celda una sola vez; mantenga las particiones durante la comparación.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X_trainval, X_test, y_trainval, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print("Características:", list(iris.feature_names))
print("Clases:", list(iris.target_names))
print("Tamaños train / validación / prueba:", len(y_train), len(y_val), len(y_test))
resultados = {}
modelos = {}

### Actividad 1. Construir y entrenar un árbol

Complete la siguiente celda utilizando `DecisionTreeClassifier`, el método `fit` y el método `predict`. Consulte `help(DecisionTreeClassifier)` si lo necesita.

1. Cree el árbol usando `profundidad` y `random_state=42`.
2. Entrénelo con `X_train` y `y_train`.
3. Obtenga las predicciones de entrenamiento y validación.
4. Use `accuracy_score(etiquetas_reales, predicciones)` para medir ambos resultados.

Ejecute primero con profundidad 2 y después con 5. Conserve las particiones y registre ambos experimentos. No utilice prueba todavía.

In [ ]:
profundidad = 2  # Segundo experimento: cambie a 5.

# TODO 1: cree el clasificador con la profundidad indicada.
modelo = ...

# TODO 2: entrene el modelo con fit (una línea).

# TODO 3: obtenga predicciones con predict.
pred_train = ...
pred_val = ...

# TODO 4: calcule las dos exactitudes con accuracy_score.
train_acc = ...
val_acc = ...

# Registro proporcionado: no necesita modificarlo.
modelos[profundidad] = modelo
resultados[profundidad] = (train_acc, val_acc)
print("Profundidad:", profundidad)
print("Exactitud de entrenamiento:", train_acc)
print("Exactitud de validación:", val_acc)

### Actividad 2. Comparar y evaluar

Complete la tabla con sus resultados:

| Profundidad | Exactitud entrenamiento | Exactitud validación |
|---|---|---|
| 2 | | |
| 5 | | |

Elija manualmente la profundidad con mejor exactitud de validación. En empate, elija 2. Complete las tres expresiones de la siguiente celda para recuperar el modelo, predecir sobre prueba y calcular su exactitud. Evalúe una sola vez y no reentrene el modelo.

In [ ]:
profundidad_elegida = ...  # Escriba 2 o 5 y justifique debajo.

# TODO 5: recupere el modelo elegido del diccionario modelos.
modelo_final = ...

# TODO 6: prediga las etiquetas de X_test con ese modelo.
predicciones = ...

# TODO 7: calcule la exactitud usando y_test y predicciones.
test_acc = ...
print("Exactitud de prueba:", test_acc)

**Análisis (máximo 5 líneas):** ¿qué modelo eligió?, ¿aumentar la profundidad ayudó?, ¿por qué un resultado perfecto en entrenamiento no garantiza generalización?

_Escriba aquí._

## 2. Consultas a un LLM con NVIDIA NIM

1. Abra [NVIDIA Build](https://build.nvidia.com/) e inicie sesión.
2. Busque el modelo indicado por el docente y abra su ejemplo de API. Obtenga una API key mediante la opción disponible en la plataforma.
3. El modelo de ejemplo es `meta/llama-3.1-8b-instruct`. Si el docente indica otro modelo habilitado en su cuenta, sustituya `MODELO` por su identificador exacto.
4. Ejecute la celda e introduzca la clave cuando aparezca el campo oculto. No la pegue en el código. Se utiliza el servicio alojado por NVIDIA, sin instalar un servidor NIM ni descargar pesos.

Referencia: [API de NVIDIA para Llama 3.1 8B](https://docs.api.nvidia.com/nim/reference/meta-llama-3_1-8b-infer).

In [ ]:
from getpass import getpass
from openai import OpenAI

MODELO = "meta/llama-3.1-8b-instruct"
api_key = getpass("API key de NVIDIA (entrada oculta): ").strip()
if not api_key:
    raise ValueError("Ingrese una API key de NVIDIA.")
cliente = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=1,
)

# Función preparada: solo necesita llamarla con el texto de su consulta.
def consultar(texto):
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Responde en español de forma clara y breve."},
            {"role": "user", "content": texto},
        ],
        temperature=0.2,
        max_tokens=350,
        stream=False,
    )
    return respuesta.choices[0].message.content

print("Cliente configurado. La primera consulta comprobará el acceso al modelo.")

### Actividad 3. Enviar una consulta propia

La función `consultar(texto)` ya realiza la llamada a NVIDIA. Complete la celda: escriba una pregunta sobre un tema de las sesiones 6–9, llame a la función con su pregunta e imprima la respuesta. No cambie la función de conexión.

In [ ]:
consulta_1 = ""  # TODO 8: escriba su pregunta.
respuesta_1 = ...  # TODO 9: llame a consultar con consulta_1.
# TODO 10: muestre respuesta_1 usando print.


### Actividad 4. Mejorar la consulta

Reformule su primera pregunta incluyendo: público principiante, un ejemplo concreto y un límite de 100 palabras. Escriba y ejecute el código necesario para enviar la nueva consulta y mostrar su respuesta. Puede apoyarse en la estructura que completó en la actividad anterior.

In [ ]:
# TODO 11: escriba consulta_2, envíela con consultar y muestre la respuesta.


### Actividad 5. Verificar una afirmación

Escriba una tercera consulta que pida evaluar esta afirmación: «Un árbol con 100 % de exactitud en entrenamiento siempre será el mejor en datos nuevos». Envíela al LLM y muestre su respuesta. Contrástela con sus resultados de sklearn y con las diapositivas; no asuma que es correcta.

In [ ]:
# TODO 12: escriba consulta_3, envíela con consultar y muestre la respuesta.


**Análisis (máximo 5 líneas):** compare las respuestas 1 y 2: claridad, ejemplo y cumplimiento del límite de palabras. ¿La respuesta 3 concuerda con lo estudiado? ¿Por qué debe verificarla? ¿Estas consultas entrenan el modelo o ejecutan inferencia?

_Escriba aquí._

### Si falla la conexión

- **401:** revise la API key y vuelva a ejecutar la configuración.
- **403/404:** revise el identificador del modelo y su acceso en NVIDIA Build.
- **429:** se alcanzó un límite de solicitudes o de uso; revise su cuenta y vuelva a intentar más tarde.
- **Timeout/5xx:** vuelva a intentar la consulta más tarde.

No incluya su clave al compartir un error. Guarde el notebook con las respuestas visibles; `getpass` no guarda el valor introducido en las celdas.

## 3. Actividad de transfer learning: use su trabajo de la sesión 8

Esta actividad se desarrolla en su notebook de datos propios, no requiere implementar aquí una nueva CNN.

Abra [el taller de la sesión 8](../../notebooks/lecture8/transfer_learning/taller_transfer_learning_datos_propios_keras3.ipynb) y retome su versión completada. Añada al final una sección titulada **«Actividad 9 — Evaluación con imágenes nuevas»**.

Debe incluir:

1. Sus tres clases, número de imágenes por clase, backbone y explicación de qué parámetros entrenó.
2. Comparación de exactitud de validación antes y después de fine-tuning, número de capas descongeladas y tasas de aprendizaje utilizadas. Use resultados existentes.
3. Seis fotografías nuevas (dos por clase) y el código de carga, preprocesamiento y predicción adaptado del notebook de clase. Respete el orden original de las clases.
4. Una tabla con clase real, predicción, probabilidad y acierto de cada imagen, junto con su visualización.
5. Análisis de dos imágenes y una mejora concreta del dataset. Si no hay errores, analice las dos predicciones menos confiadas.

Escriba usted las llamadas para cargar las imágenes y ejecutar su modelo, apoyándose en el material de clase. No se proporciona la solución en este notebook. No reentrene usando estas seis imágenes ni cambie el modelo según sus resultados.

Adjunte su notebook de transfer learning junto con este notebook. El enunciado contiene la tabla y los criterios completos. Si perdió el modelo, deberá recuperar el flujo de entrenamiento del trabajo anterior; eso agrega tiempo a la actividad.
